# 5 · Hybrid Models

- **Linear regression** extrapolates trends but can't learn interactions.
- **XGBoost / trees** learn interactions but can't extrapolate. Their predictions stay inside the range of the training targets.

A **boosted hybrid** gets both: model 1 learns the trend, and model 2 learns the residuals.

```
series = trend + seasons + cycles + error
          ↑ model_1 ↑   ↑ model_2 on (y − ŷ₁) ↑
```

```python
model_1.fit(X_1, y)
y_resid = y - model_1.predict(X_1)
model_2.fit(X_2, y_resid)
y_pred = model_1.predict(X_1) + model_2.predict(X_2)
```
Give each model its own features. `X_1` holds what LR should extrapolate (the time trend). `X_2` holds what trees learn well
(categories, day of month, promotions).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

family_sales = (
    store_sales
    .groupby(["family", "date"], observed=True).mean()
    .unstack("family")
    .loc["2017"]
)

## 1. Model 1: trend with linear regression (wide format, one column per family)

In [ ]:
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import DeterministicProcess

y = family_sales.loc[:, "sales"]

dp = DeterministicProcess(index=y.index, constant=True, order=1, drop=True)
X_1 = dp.in_sample()

model_1 = LinearRegression(fit_intercept=False).fit(X_1, y)
y_fit_1 = pd.DataFrame(model_1.predict(X_1), index=y.index, columns=y.columns)

## 2. Model 2: XGBoost on residuals (long format)
XGBoost predicts one output, so **stack** the series into rows and label-encode which series each row belongs to.

In [ ]:
from xgboost import XGBRegressor

X_2 = family_sales.drop("sales", axis=1).stack()     # wide → long: one row per (date, family)
X_2 = X_2.reset_index("family")
X_2["family"], _ = X_2["family"].factorize()         # label-encode the series id
X_2["day"] = X_2.index.day                           # day of month (seasonality for trees)

y_resid = (y - y_fit_1).stack()                      # residuals in the same long order

model_2 = XGBRegressor().fit(X_2, y_resid)
X_2.head()

## 3. Combine

In [ ]:
y_fit = y_fit_1.stack() + model_2.predict(X_2)       # trend + learned residuals
y_fit = y_fit.unstack().clip(0.0)                    # back to wide

families = y.columns[:4]
axs = y[families].plot(subplots=True, sharex=True, figsize=(11, 7), **plot_params, alpha=0.5)
y_fit[families].plot(subplots=True, sharex=True, color="C0", ax=axs)
for ax, family in zip(axs, families):
    ax.legend([]); ax.set_ylabel(family)

## Reusable `BoostedHybrid` class (from the course)

In [ ]:
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2

    def fit(self, X_1, X_2, y):
        self.model_1.fit(X_1, y)
        y_fit = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=y.columns)
        y_resid = (y - y_fit).stack()
        self.model_2.fit(X_2, y_resid)
        self.y_columns = y.columns
        return self

    def predict(self, X_1, X_2):
        y_pred = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=self.y_columns)
        y_pred = y_pred.stack() + self.model_2.predict(X_2)
        return y_pred.unstack()

Validate on a time split. Then try other pairs: model 1 from the linear family, model 2 anything non-linear.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_log_error

y_train, y_valid = y.loc[:"2017-07-01"], y.loc["2017-07-02":]
X1_train, X1_valid = X_1.loc[:"2017-07-01"], X_1.loc["2017-07-02":]
X2_train, X2_valid = X_2.loc[:"2017-07-01"], X_2.loc["2017-07-02":]

for m1, m2 in [
    (LinearRegression(), XGBRegressor()),
    (Ridge(), RandomForestRegressor(n_estimators=100)),
    (Ridge(), KNeighborsRegressor()),
]:
    model = BoostedHybrid(m1, m2).fit(X1_train, X2_train, y_train)
    y_pred = model.predict(X1_valid, X2_valid).clip(0.0)
    score = mean_squared_log_error(y_valid, y_pred[y_valid.columns]) ** 0.5
    print(f"{type(m1).__name__:>16} + {type(m2).__name__:<22} valid RMSLE {score:.4f}")

**Takeaways**
- The usual design is a simple linear model for the trend followed by a powerful non-linear model on the residuals.
- The hybrid's trend can only be as good as model 1's. Model 2 can't fix a badly fitted trend.
- A **stacked** hybrid is the alternative: feed model 1's predictions to model 2 as a feature.